# Gene Analysis Notebook

This notebook loads the generated Mytho-Toon Genome using PySpark and displays the results.

Use the super pyenv to run this notebook; check README.md for details.


In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, size

# Ensure we can import our modules
# sys.path.append(os.path.abspath("super-services/src"))

from super.core.runtime import *

from super.core import utils
from super.apps.generate_powers.models import MutatedGene

In [ ]:
# Initialize Spark Session
bootstrap_spark_env()

# set memory above default
spark = (SparkSession.builder
.appName("VCP Low-Level Catalog")
.config("spark.executor.memory", "16g")
.config("spark.driver.memory", "16g")
.getOrCreate())

In [ ]:

# Load App Configuration to find paths
conf = utils.get_app_conf("generate_powers")
stage_root = conf.get_string("stage_root")

print(f"Project Stage Root: {stage_root}")

In [ ]:
# Defined path to genes
genes_path = os.path.join(stage_root, "generated_genome", "*", "*.json")
print(f"Reading genes from: {genes_path}")

# Load JSONs into DataFrame
df = spark.read.option("multiline", "true").json(genes_path)

# Show Schema
df.printSchema()

In [ ]:
# Analysis: Show top level fields
display_df = df.select(
    col("gene_id"),
    col("gene_role"),
    col("mutation_class"),
    col("failure_mode"),
    col("confidence"),
    size(col("regulated_genes")).alias("num_links"),
    size(col("side_effect_profile")).alias("num_side_effects")
)

df.show(20, truncate=False)

In [ ]:
# Count by Role
df.groupBy("gene_role").count().show()